## Extracting the data from db

In [1]:
import mysql.connector
import pandas as pd

# Database connection details
HOST = "localhost"
USER = "root"
PASSWORD = "root"
DATABASE = "eco_friendly"

try:
    # Connect to MySQL database
    connection = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD,
        database=DATABASE
    )

    if connection.is_connected():
        print("✅ Connected to MySQL Database")

        # Query to fetch all data from the Products table
        query = "SELECT * FROM product"

        # Load data into a Pandas DataFrame
        df = pd.read_sql(query, connection)

        print("✅ Data successfully extracted from MySQL")
        display(df.head())  # Display first 5 rows

except mysql.connector.Error as e:
    print(f"❌ Error: {e}")

finally:
    if connection.is_connected():
        connection.close()
        print("✅ MySQL connection is closed")

✅ Connected to MySQL Database
✅ Data successfully extracted from MySQL


C:\Users\DELL\AppData\Local\Temp\ipykernel_13112\1788276798.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,product_id,user_id,product_name,category,material,price,description,brand,availability,ratings
0,1,100,Bamboo Toothbrush,Toothbrush,Bamboo,45.67,Eco-friendly Toothbrush made from Bamboo. Sust...,EcoBrand,In Stock,8.36
1,2,101,Recycled Plastic Toothbrush,Toothbrush,Recycled Plastic,24.41,Eco-friendly Toothbrush made from Recycled Pla...,EcoBrand,In Stock,7.98
2,3,102,Cornstarch Toothbrush,Toothbrush,Cornstarch,16.19,Eco-friendly Toothbrush made from Cornstarch. ...,EcoBrand,Out of Stock,8.89
3,4,103,Silicone Toothbrush,Toothbrush,Silicone,48.35,Eco-friendly Toothbrush made from Silicone. Su...,EcoBrand,Out of Stock,7.21
4,5,104,Charcoal-infused Toothbrush,Toothbrush,Charcoal-infused,15.17,Eco-friendly Toothbrush made from Charcoal-inf...,EcoBrand,In Stock,9.01


✅ MySQL connection is closed


## Content based filtering

In [2]:
df.columns

Index(['product_id', 'user_id', 'product_name', 'category', 'material',
       'price', 'description', 'brand', 'availability', 'ratings'],
      dtype='object')

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Rename column if needed
#df.rename(columns={"User-ID": "user_id"}, inplace=True)

# Step 2: Get Unique Categories & Ask User to Select One
unique_categories = df['category'].unique()
print("Available Categories:", unique_categories)

selected_category = input("Enter the category you want recommendations from: ").strip()

# Step 3: Filter Products by Selected Category
df_filtered = df[df['category'] == selected_category].reset_index(drop=True)

# Step 4: Combine 'category', 'material', and 'description' into a single feature
df_filtered['combined_features'] = df_filtered['category'] + " " + df_filtered['material'] + " " + df_filtered['description']

# Step 5: Convert text data into numerical form (TF-IDF Vectorization)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_filtered['combined_features'])

# Step 6: Compute Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Step 7: Create Recommendation Function
def recommend_products(product_name, num_recommendations=5):
    # Check if the product exists in the selected category
    if product_name not in df_filtered['product_name'].values:
        print("Product not found in the selected category.")
        return None
    
    # Find index of the product
    idx = df_filtered[df_filtered['product_name'] == product_name].index[0]
    
    # Get similarity scores for all products in the selected category
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort products by similarity score (highest first)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top N similar products (excluding itself)
    sim_scores = sim_scores[1:num_recommendations+1]
    
    # Get recommended product indices
    product_indices = [i[0] for i in sim_scores]
    
    # Return top recommended products with all requested columns
    return df_filtered[['product_id', 'user_id', 'product_name', 'category', 'material',
                        'price', 'description', 'brand', 'availability', 'ratings']].iloc[product_indices]

# Step 8: Ask User for a Product Name in the Selected Category
print("\nAvailable Products in Category:", selected_category)
print(df_filtered['product_name'].tolist())

product_to_search = input("Enter the product name for recommendations: ").strip()

# Step 9: Get Recommendations
recommendations = recommend_products(product_to_search, 5)

if recommendations is not None:
    print("\nRecommended Products:")
    display(recommendations)


Available Categories: ['Toothbrush' 'Bag' 'Water Bottle' 'Clothing' 'Cutlery' 'Notebook' 'Shoes'
 'Toys' 'Straws' 'Phone Cases']


Enter the category you want recommendations from:  Straws



Available Products in Category: Straws
['Bamboo Straws', 'Wheat Straw Straws', 'Stainless Steel Straws', 'Glass Straws', 'Silicone Straws', 'Rice Straws', 'Sugarcane Straws', 'Paper Straws', 'Cornstarch Straws', 'Reed Straws']


Enter the product name for recommendations:  Reed Straws



Recommended Products:


,product_id,user_id,product_name,category,material,price,description,brand,availability,ratings
0,81,180,Bamboo Straws,Straws,Bamboo,28.18,Eco-friendly Straws made from Bamboo. Sustaina...,EcoBrand,In Stock,7.63
3,84,183,Glass Straws,Straws,Glass,5.37,Eco-friendly Straws made from Glass. Sustainab...,EcoBrand,Out of Stock,5.47
4,85,184,Silicone Straws,Straws,Silicone,39.23,Eco-friendly Straws made from Silicone. Sustai...,EcoBrand,In Stock,7.45
5,86,185,Rice Straws,Straws,Rice,30.79,Eco-friendly Straws made from Rice. Sustainabl...,EcoBrand,Out of Stock,7.87
6,87,186,Sugarcane Straws,Straws,Sugarcane,12.93,Eco-friendly Straws made from Sugarcane. Susta...,EcoBrand,Out of Stock,6.43


## Collobarative filtering

In [8]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jaccard
import ipywidgets as widgets
from IPython.display import display, clear_output


# Step 1: User ID Dropdown (simulated)
user_id_dropdown = widgets.IntText(
    value=1,
    description="User ID:",
    disabled=False
)

# Step 2: Dropdown for Product Selection
product_dropdown = widgets.Dropdown(
    options=df['product_name'].unique(),
    description="Select Product:",
    disabled=False
)

# Step 3: Button to Trigger Recommendations
recommend_button = widgets.Button(
    description="Get Recommendations",
    button_style="primary"
)

# Output widget to display results
output = widgets.Output()

# Function to Calculate Jaccard Similarity
def calculate_jaccard_similarity(df):
    # Ensure 'price' and 'ratings' are numeric
    df[['price', 'ratings']] = df[['price', 'ratings']].apply(pd.to_numeric, errors='coerce')

    # Convert to binary (above/below median)
    binary_features = np.where(df[['price', 'ratings']] > df[['price', 'ratings']].median(), 1, 0)
    
    # Compute Jaccard similarity matrix
    num_products = len(df)
    sim_matrix = np.zeros((num_products, num_products))

    for i in range(num_products):
        for j in range(num_products):
            if i != j:
                sim_matrix[i][j] = 1 - jaccard(binary_features[i], binary_features[j])

    return sim_matrix

# Function to Get Recommendations
def recommend_products(user_id, product_name, num_recommendations=5):
    with output:
        clear_output()
        
        # Check if product exists
        if product_name not in df['product_name'].values:
            print("\n⚠️ Product not found in dataset.")
            return
        
        # Compute Jaccard similarity
        similarity_matrix = calculate_jaccard_similarity(df)

        # Get index of selected product
        product_idx = df[df['product_name'] == product_name].index[0]

        # Find similar products
        sim_scores = list(enumerate(similarity_matrix[product_idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:num_recommendations + 1]  # Exclude the selected product itself

        # Get recommended product indices
        product_indices = [i[0] for i in sim_scores]

        print("\n🔹 Recommended Products Based on:", product_name, "(Jaccard Similarity)")
        display(df.iloc[product_indices])

# Function to Handle Button Click
def on_button_click(b):
    selected_product = product_dropdown.value
    user_id = user_id_dropdown.value  # Get user_id from widget
    if selected_product:
        recommend_products(user_id, selected_product, 5)

# Link button to function
recommend_button.on_click(on_button_click)

# Display widgets
display(user_id_dropdown, product_dropdown, recommend_button, output)


IntText(value=1, description='User ID:')

Dropdown(description='Select Product:', options=('Bamboo Toothbrush', 'Recycled Plastic Toothbrush', 'Cornstar…

Button(button_style='primary', description='Get Recommendations', style=ButtonStyle())

Output()

## Hybrid filtering

In [7]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jaccard
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output



# User Input Widgets
user_id_dropdown = widgets.IntText(value=1, description="User ID:", disabled=False)
product_dropdown = widgets.Dropdown(options=df['product_name'].unique(), description="Select Product:", disabled=False)
recommend_button = widgets.Button(description="Get Recommendations", button_style="primary")
output = widgets.Output()

# Function to Compute Jaccard Similarity
def calculate_jaccard_similarity(df):
    df[['price', 'ratings']] = df[['price', 'ratings']].apply(pd.to_numeric, errors='coerce')
    binary_features = np.where(df[['price', 'ratings']] > df[['price', 'ratings']].median(), 1, 0)
    
    num_products = len(df)
    sim_matrix = np.zeros((num_products, num_products))
    for i in range(num_products):
        for j in range(num_products):
            if i != j:
                sim_matrix[i][j] = 1 - jaccard(binary_features[i], binary_features[j])
    
    return sim_matrix

# Function to Compute Content Similarity
def calculate_content_similarity(df):
    df['combined_features'] = df['category'] + " " + df['material'] + " " + df['description']
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df['combined_features'])
    return cosine_similarity(tfidf_matrix, tfidf_matrix)

# Hybrid Recommendation Function
def recommend_products(user_id, product_name, num_recommendations=5):
    with output:
        clear_output()
        
        if product_name not in df['product_name'].values:
            print("⚠️ Product not found in dataset.")
            return
        
        jaccard_matrix = calculate_jaccard_similarity(df)
        content_matrix = calculate_content_similarity(df)
        
        product_idx = df[df['product_name'] == product_name].index[0]
        
        hybrid_scores = (jaccard_matrix[product_idx] + content_matrix[product_idx]) / 2
        sim_scores = list(enumerate(hybrid_scores))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:num_recommendations + 1]
        
        product_indices = [i[0] for i in sim_scores]
        
        print("🔹 Recommended Products Based on:", product_name)
        display(df.iloc[product_indices])

# Button Click Function
def on_button_click(b):
    selected_product = product_dropdown.value
    user_id = user_id_dropdown.value
    if selected_product:
        recommend_products(user_id, selected_product, 5)

recommend_button.on_click(on_button_click)

# Display Widgets
display(user_id_dropdown, product_dropdown, recommend_button, output)


IntText(value=1, description='User ID:')

Dropdown(description='Select Product:', options=('Bamboo Toothbrush', 'Recycled Plastic Toothbrush', 'Cornstar…

Button(button_style='primary', description='Get Recommendations', style=ButtonStyle())

Output()